# Day 3-01｜兩張畫面中的框，哪一些屬於同一位球員？IoU 關聯

> Day 2 已經能在每個 frame 找到球員，但模型尚不知道 `Frame 15` 的某個框與 `Frame 16` 的哪個框是同一人。  
> 我們先用最簡單、可完全看懂的 IoU 配對建立 tracking 直覺，再於下一單元交給 ByteTrack 處理長影片。

## 我們會完成什麼

- 為前後兩張圖的框命名為 `A0, A1, ...` 與 `B0, B1, ...`。
- 手動拆解交集、聯集與 IoU，建立所有框的兩兩比較矩陣。
- 用 heatmap 看清楚每個 A 框和所有 B 框的關係。
- 用全域 greedy 規則建立一對一配對，最後把配對線畫回影像。

## 先認識本單元名詞

- **Association（關聯）**：判斷不同 frame 的兩筆 detection 是否屬於同一目標。
- **IoU（Intersection over Union）**：兩個矩形的交集面積除以聯集面積，範圍為 0 到 1。
- **Greedy（貪婪法）**：每次先選目前分數最高且不衝突的候選配對。


## 本單元定位

- 我們只比較相鄰兩個 frame，目的是看懂關聯過程，不是重做完整 ByteTrack。
- 這一格運算量不大，CPU 就能完成；若 YOLO 推論較慢，可降低 `IMGSZ`。
- 請同時看「影像、矩陣、配對線」，不要只看最後的 assignment 表格。


## 工作坊流程

1. 對相鄰兩個 frame 執行 detector，只保留球員框。
2. 在圖上標出 `A0...` 與 `B0...`，讓矩陣的列與欄能回到真實框。
3. 計算所有 `A_i` 與 `B_j` 的 IoU，畫成 heatmap。
4. 將所有候選依 IoU 由大到小排序，建立不重複的一對一配對。
5. 把配對線畫回兩張圖，討論快速移動、遮擋與框抖動為何會失敗。


In [1]:
# 這一格要做什麼：定位 repo、準備課程環境並載入共用模組。
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


課程根目錄: /home/nckusoc/repos/basketball-hackathon-course
素材資料夾: /home/nckusoc/repos/basketball-hackathon-course/assets
工具模組: /home/nckusoc/repos/basketball-hackathon-course/src


## Step 1｜選擇影片、模型與相鄰 frame

`FRAME_B = FRAME_A + 1` 表示我們只跨一個 frame。兩張圖時間越接近，同一位球員的框通常重疊越多，IoU 關聯也越容易成立。


In [ ]:
# 這一格要做什麼：設定資料來源與實驗參數。
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch, Rectangle
import numpy as np
import pandas as pd

from src.yolo_utils import (
    PLAYER_CLASS_NAMES,
    detector_model_path,
    first_reference_video,
    read_video_frame,
    run_detector_on_image,
)

VIDEO_PATH = first_reference_video(COURSE_ROOT)
MODEL_PATH = detector_model_path(COURSE_ROOT)
FRAME_A = 15
FRAME_B = FRAME_A + 1  # 只比較下一個 frame，先把問題簡化。
CONF = 0.25            # 低於此 confidence 的 detection 不會保留。
IMGSZ = 960            # 推論影像尺寸；較小會較快，但小物件可能更難偵測。

print("video:", VIDEO_PATH)
print("model:", MODEL_PATH)
print("frames:", FRAME_A, "->", FRAME_B)


## Step 2｜先把每個偵測框編號

`A2` 不是球員的永久身分，只表示「Frame A 的第 3 個框」；`B2` 也只是 Frame B 的第 3 個框。真正的跨影格身分要等關聯後才能建立。


In [ ]:
# 這一格要做什麼：讀取兩張圖、執行偵測，並只留下 player 類別。
frame_a = read_video_frame(VIDEO_PATH, FRAME_A)
frame_b = read_video_frame(VIDEO_PATH, FRAME_B)

dets_a, _ = run_detector_on_image(
    MODEL_PATH, frame_a, conf=CONF, imgsz=IMGSZ, frame_index=FRAME_A
)
dets_b, _ = run_detector_on_image(
    MODEL_PATH, frame_b, conf=CONF, imgsz=IMGSZ, frame_index=FRAME_B
)

players_a = [det for det in dets_a if det.class_name in PLAYER_CLASS_NAMES]
players_b = [det for det in dets_b if det.class_name in PLAYER_CLASS_NAMES]

print(f"Frame A 有 {len(players_a)} 個 player boxes")
print(f"Frame B 有 {len(players_b)} 個 player boxes")


In [ ]:
# 這一格要做什麼：將矩陣標籤 A0/B0 畫回真正的偵測框。
def draw_indexed_boxes(ax, image, detections, prefix, title):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")
    for index, det in enumerate(detections):
        x1, y1, x2, y2 = det.bbox_xyxy
        ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1,
                               fill=False, edgecolor="#00E5FF", linewidth=2.5))
        ax.text(x1, max(0, y1 - 6), f"{prefix}{index}", color="black",
                fontsize=11, weight="bold",
                bbox={"facecolor": "#00E5FF", "alpha": 0.9, "pad": 2})

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
draw_indexed_boxes(axes[0], frame_a, players_a, "A", f"Frame A = {FRAME_A}")
draw_indexed_boxes(axes[1], frame_b, players_b, "B", f"Frame B = {FRAME_B}")
plt.tight_layout()
plt.show()


### 如何讀上圖

稍後矩陣中的「第 `A1` 列、第 `B3` 欄」，就是在問：**左圖 A1 的框與右圖 B3 的框有多少比例重疊？** 先建立這個圖像對照，矩陣就不會只是一堆抽象數字。


## Step 3｜把 IoU 拆成可檢查的幾何步驟

若框 $A=(a_{x1},a_{y1},a_{x2},a_{y2})$、框 $B=(b_{x1},b_{y1},b_{x2},b_{y2})$，先取兩框交集矩形：

$$
x_{\mathrm{left}}=\max(a_{x1},b_{x1}),\quad
y_{\mathrm{top}}=\max(a_{y1},b_{y1})
$$

$$
x_{\mathrm{right}}=\min(a_{x2},b_{x2}),\quad
y_{\mathrm{bottom}}=\min(a_{y2},b_{y2})
$$

接著計算

$$
\mathrm{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}
=\frac{\text{intersection}}{\text{area}_A+\text{area}_B-\text{intersection}}.
$$

- `0`：完全沒有重疊。
- 接近 `1`：兩框幾乎位於相同位置。
- IoU 只看空間重疊，不知道球衣顏色、移動方向或球員外觀。


In [ ]:
# 這一格要做什麼：逐步計算單一 pair 的 IoU，再建立所有 A/B 組合的矩陣。
def iou(box_a, box_b):
    '''Return the intersection-over-union of two [x1, y1, x2, y2] boxes.'''
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    # 交集矩形的左上角要取較大的座標，右下角要取較小的座標。
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    intersection_width = max(0.0, ix2 - ix1)
    intersection_height = max(0.0, iy2 - iy1)
    intersection = intersection_width * intersection_height

    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0

# matrix[i, j] 表示 A_i 與 B_j 的 IoU；每個 A 框都會和每個 B 框比較。
matrix = np.zeros((len(players_a), len(players_b)), dtype=float)
for i, det_a in enumerate(players_a):
    for j, det_b in enumerate(players_b):
        matrix[i, j] = iou(det_a.bbox_xyxy, det_b.bbox_xyxy)

matrix_df = pd.DataFrame(
    np.round(matrix, 3),
    index=[f"A{i}" for i in range(len(players_a))],
    columns=[f"B{j}" for j in range(len(players_b))],
)
matrix_df


In [ ]:
# 這一格要做什麼：用顏色深淺顯示所有偵測框之間的關係。
fig, ax = plt.subplots(figsize=(max(6, len(players_b) * 0.8), max(4, len(players_a) * 0.65)))
heatmap = ax.imshow(matrix, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(players_b)), labels=[f"B{j}" for j in range(len(players_b))])
ax.set_yticks(range(len(players_a)), labels=[f"A{i}" for i in range(len(players_a))])
ax.set_xlabel("Frame B detections")
ax.set_ylabel("Frame A detections")
ax.set_title("Pairwise IoU：每個 A 框與每個 B 框的重疊關係")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        value = matrix[i, j]
        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                color="white" if value > 0.55 else "black")
fig.colorbar(heatmap, ax=ax, label="IoU (0=no overlap, 1=same box)")
plt.tight_layout()
plt.show()


## Step 4｜先列出每個 A 框的最佳候選

每一列最大值回答「這個 A 框最像哪一個 B 框？」但它還不是最終配對，因為不同 A 框可能同時把同一個 B 框當成最佳答案。一對一限制會在下一步處理。


In [ ]:
# 這一格要做什麼：只做候選診斷，尚未套用一對一限制。
best_match_rows = []
for i in range(matrix.shape[0]):
    if matrix.shape[1] == 0:
        break
    j = int(matrix[i].argmax())
    best_match_rows.append({
        "frame_a_box": f"A{i}",
        "best_frame_b_box": f"B{j}",
        "best_iou": float(matrix[i, j]),
        "passes_threshold": bool(matrix[i, j] >= 0.30),
    })

best_match_df = pd.DataFrame(best_match_rows)
best_match_df


## Step 5｜用全域 greedy 規則建立一對一配對

1. 列出所有高於門檻的 `(A_i, B_j)` 候選。
2. 依 IoU 從大到小排序。
3. 每次取目前最高分；若 A 或 B 已被使用，就跳過。
4. 直到沒有候選可選。

這仍不是 ByteTrack，但比逐列直接決定更公平：不會因為 `A0` 比 `A1` 先出現在迴圈，就搶走其實更適合 `A1` 的 B 框。


In [ ]:
# 這一格要做什麼：從所有 pair 中，依分數建立不衝突的一對一配對。
IOU_THRESHOLD = 0.30
candidates = [
    (float(matrix[i, j]), i, j)
    for i in range(matrix.shape[0])
    for j in range(matrix.shape[1])
    if matrix[i, j] >= IOU_THRESHOLD
]
candidates.sort(reverse=True)  # Python tuple 會先依第一個欄位（IoU）排序。

assignments = []
used_a, used_b = set(), set()
for score, i, j in candidates:
    if i in used_a or j in used_b:
        continue
    used_a.add(i)
    used_b.add(j)
    assignments.append({
        "frame_a_box": f"A{i}",
        "frame_b_box": f"B{j}",
        "iou": round(score, 3),
        "a_index": i,
        "b_index": j,
    })

assignment_df = pd.DataFrame(assignments)
assignment_df[["frame_a_box", "frame_b_box", "iou"]] if not assignment_df.empty else assignment_df


In [ ]:
# 這一格要做什麼：把 assignment 表格中的關係畫回兩張圖。
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
draw_indexed_boxes(axes[0], frame_a, players_a, "A", f"Frame {FRAME_A}")
draw_indexed_boxes(axes[1], frame_b, players_b, "B", f"Frame {FRAME_B}")

# 透過公開 API 取得 colormap，避免型別檢查器把動態屬性 `plt.cm.tab10` 視為未知。
colors = plt.get_cmap("tab10")(np.linspace(0, 1, max(1, len(assignments))))
for color, pair in zip(colors, assignments):
    box_a = players_a[pair["a_index"]].bbox_xyxy
    box_b = players_b[pair["b_index"]].bbox_xyxy
    center_a = ((box_a[0] + box_a[2]) / 2, (box_a[1] + box_a[3]) / 2)
    center_b = ((box_b[0] + box_b[2]) / 2, (box_b[1] + box_b[3]) / 2)
    connection = ConnectionPatch(
        xyA=center_b, coordsA=axes[1].transData,
        xyB=center_a, coordsB=axes[0].transData,
        color=color, linewidth=2.5, alpha=0.9,
    )
    fig.add_artist(connection)
    axes[0].text(*center_a, f" IoU={pair['iou']:.2f}", color=color, weight="bold")

fig.suptitle("Greedy assignments：同色連線代表被視為同一位球員", fontsize=14)
plt.tight_layout()
plt.show()


## 從簡化 IoU 走向 ByteTrack

純 IoU 在球員快速移動、彼此交錯、被遮擋或 detector 漏偵時容易失敗。ByteTrack 會維護既有 track 狀態，並分兩階段使用高、低 confidence detections，嘗試讓短暫不確定的目標仍能接回原 ID。下一單元會直接觀察 `track_id` 與軌跡。

### 請你回答

1. heatmap 中是否有一列同時出現兩個相近的高分？這代表什麼不確定性？
2. 把 `IOU_THRESHOLD` 改成 `0.10` 或 `0.60`，配對數量如何改變？
3. 哪一種籃球畫面情況會讓正確配對的 IoU 反而很低？


## 本單元產出

- 有 A/B 編號的前後 frame 對照圖。
- 所有 detection pairs 的 IoU heatmap。
- 一對一 assignment 表格與跨圖配對線。

這一單元刻意不輸出影片；我們先把兩個 frame 的關係看清楚，再進入長時間追蹤。
